# H2-1 extension, step 7: multiple-comparison correction (Benjamini-Hochberg)

**Summary of the step 6 results** (raw p-values)
- Significant (p<0.05): TypeC x high (coef=-6.25, p=0.0004), TypeB x high (coef=-2.43, p=0.0118)
- The other 7 combinations: not significant (TypeC x low has a large coefficient but n=28 and p=0.163, so it may be noise)

**This step**: the 9 type x tier combinations were tested together, so calling an individual p-value "significant" as is
raises the multiple-comparison problem (a higher chance of false positives). Apply the Benjamini-Hochberg (FDR) correction
to settle "how many of the 9 survive the correction".

**Expectation (mental arithmetic while interpreting step 6)**: only TypeC x high survives, and TypeB x high may narrowly drop out.
This step computes it exactly to confirm.

## 0. Environment setup

In [1]:
from importlib.metadata import version
print("linearmodels", version("linearmodels"), "| statsmodels", version("statsmodels"))

linearmodels 7.0 | statsmodels 0.14.6


In [2]:
import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS
from statsmodels.stats.multitest import multipletests

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

from config import (DATA_DIR, OUT_DIR, INTERMEDIATE_DIR, FIG_DIR, SEED, TIER_MIN_WEEK, TIER_MAX_WEEK,
                    CAMP_MIN_WEEK, CAMP_MAX_WEEK, MID_WEEK, CELL_SIZE_THRESH, TIER_EN,
                    load_tiers, load_panel, load_camp_full)



## Rebuild the step 1-6 pipeline (same logic, nothing new)

In [3]:
panel = load_panel()
tier_df = load_tiers()

# --- Type x tier interaction dummies ---
for t in ["TypeA", "TypeB", "TypeC"]:
    for tier in ["low", "mid", "high"]:
        panel[f"{t}_x_{tier}"] = ((panel[f"active_{t}"] == 1) & (panel["tier3"] == tier)).astype(int)

interaction_cols = [f"{t}_x_{tier}" for t in ["TypeA", "TypeB", "TypeC"] for tier in ["low", "mid", "high"]]

# --- Cell-size table (step 4) ---
def cell_size_table(tier_col):
    out = {}
    for t in ["TypeA", "TypeB", "TypeC"]:
        hh_active = panel.loc[panel[f"active_{t}"] == 1, ["household_key", tier_col]].drop_duplicates()
        out[t] = hh_active.groupby(tier_col, observed=True)["household_key"].nunique()
    return pd.DataFrame(out)
cell3 = cell_size_table("tier3")

# --- Step 6 main regression ---
reg_df = panel.copy()
reg_df["entity"] = reg_df["household_key"]
reg_df["time"] = reg_df["WEEK_NO"]
reg_df = reg_df.set_index(["entity", "time"])

model = PanelOLS(
    dependent=reg_df["spend"],
    exog=reg_df[interaction_cols],
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
result = model.fit(cov_type="clustered", cluster_entity=True)

print("Panel loaded + step 6 regression re-run done")
print(result.params)

Panel loaded + step 6 regression re-run done
TypeA_x_low     1.056302
TypeA_x_mid     0.819589
TypeA_x_high   -0.162506
TypeB_x_low     1.442018
TypeB_x_mid     0.417724
TypeB_x_high   -2.429732
TypeC_x_low     9.675478
TypeC_x_mid    -1.219311
TypeC_x_high   -6.252591
Name: parameter, dtype: float64


## Step 7: multiple-comparison correction (Benjamini-Hochberg / FDR)

The p-values of the 9 combinations were tested together, so significance is not judged by an individual p<0.05 alone;
it is corrected at a 5% FDR (false discovery rate).

In [4]:
coef_table = result.params.to_frame("coef")
coef_table["se"] = result.std_errors
coef_table["t"] = result.tstats
coef_table["p_raw"] = result.pvalues
coef_table = coef_table.reset_index().rename(columns={"index": "variable"})
coef_table[["Type", "tier"]] = coef_table["variable"].str.split("_x_", expand=True)

# Merge the cell sizes
cell_long = cell3.stack().reset_index()
cell_long.columns = ["tier", "Type", "n_recipients"]
coef_table = coef_table.merge(cell_long, on=["Type", "tier"], how="left")
coef_table["small_cell"] = coef_table["n_recipients"] < CELL_SIZE_THRESH

# --- BH (FDR) correction ---
reject, p_adj, _, _ = multipletests(coef_table["p_raw"], alpha=0.05, method="fdr_bh")
coef_table["p_adj_bh"] = p_adj
coef_table["reject_bh"] = reject

_pt = pd.read_csv(OUT_DIR / "pretrend_tests.csv").set_index("scope")
pretrend_note = {t: f"step 5 p={_pt.loc[TIER_EN[t], 'p_value']:.3f} ({int(_pt.loc[TIER_EN[t], 'n_never_receive']):,} non-recipient households)" for t in ["low", "mid", "high"]}
coef_table["pretrend_note"] = coef_table["tier"].map(pretrend_note)

cols_order = ["Type", "tier", "coef", "se", "p_raw", "p_adj_bh", "reject_bh", "n_recipients", "small_cell", "pretrend_note"]
final_table = coef_table[cols_order].sort_values("p_raw", kind="stable")
print(final_table.to_string(index=False))


 Type tier      coef       se    p_raw  p_adj_bh  reject_bh  n_recipients  small_cell                                 pretrend_note
TypeC high -6.252591 1.756420 0.000371  0.003341       True           295       False  step 5 p=0.613 (49 non-recipient households)
TypeB high -2.429732 0.965271 0.011832  0.053243      False           628       False  step 5 p=0.613 (49 non-recipient households)
TypeC  low  9.675478 6.939703 0.163253  0.489758      False            28        True step 5 p=0.764 (624 non-recipient households)
TypeA  mid  0.819589 0.687613 0.233289  0.524901      False           554       False step 5 p=0.053 (243 non-recipient households)
TypeA  low  1.056302 1.431923 0.460710  0.805521      False           187       False step 5 p=0.764 (624 non-recipient households)
TypeB  low  1.442018 2.335870 0.537014  0.805521      False            90       False step 5 p=0.764 (624 non-recipient households)
TypeB  mid  0.417724 1.362253 0.759116  0.833748      False           305   

In [5]:
print("="*60)
print("Before vs after correction")
print("="*60)
n_sig_raw = (coef_table["p_raw"] < 0.05).sum()
n_sig_bh = coef_table["reject_bh"].sum()
print(f"Combinations significant at raw p<0.05: {n_sig_raw}")
print(f"Combinations still significant after BH correction: {n_sig_bh}")
print()
print("[Combinations that survive the BH correction]")
survivors = final_table[final_table["reject_bh"]]
print(survivors.to_string(index=False) if len(survivors) > 0 else "None")
print()
print("[Combinations significant before the correction that drop out after it]")
dropped = final_table[(final_table["p_raw"] < 0.05) & (~final_table["reject_bh"])]
print(dropped.to_string(index=False) if len(dropped) > 0 else "None")


Before vs after correction
Combinations significant at raw p<0.05: 2
Combinations still significant after BH correction: 1

[Combinations that survive the BH correction]
 Type tier      coef      se    p_raw  p_adj_bh  reject_bh  n_recipients  small_cell                                pretrend_note
TypeC high -6.252591 1.75642 0.000371  0.003341       True           295       False step 5 p=0.613 (49 non-recipient households)

[Combinations significant before the correction that drop out after it]
 Type tier      coef       se    p_raw  p_adj_bh  reject_bh  n_recipients  small_cell                                pretrend_note
TypeB high -2.429732 0.965271 0.011832  0.053243      False           628       False step 5 p=0.613 (49 non-recipient households)


In [6]:
# --- source CSV: the full nine-row table with Benjamini-Hochberg adjustment
_ci = result.conf_int()
_bh = coef_table.copy()
_bh["ci_low"] = [_ci.loc[f"{r.Type}_x_{r.tier}", "lower"] for r in _bh.itertuples()]
_bh["ci_high"] = [_ci.loc[f"{r.Type}_x_{r.tier}", "upper"] for r in _bh.itertuples()]
_bh["bh_rank"] = _bh["p_raw"].rank(method="first").astype(int)
_bh["bh_threshold"] = 0.05 * _bh["bh_rank"] / len(_bh)
_bh["pretrend_p"] = _bh["tier"].map(lambda t: _pt.loc[TIER_EN[t], "p_value"])
_bh["pretrend_n_control"] = _bh["tier"].map(lambda t: int(_pt.loc[TIER_EN[t], "n_never_receive"]))
_bh["tier"] = _bh["tier"].map(TIER_EN)
_bh.rename(columns={"Type": "type", "t": "t_stat"}) \
    [["type", "tier", "coef", "se", "ci_low", "ci_high", "t_stat", "p_raw", "p_adj_bh", "reject_bh", "bh_rank", "bh_threshold",
      "n_recipients", "small_cell", "pretrend_p", "pretrend_n_control"]] \
    .sort_values("p_raw", kind="stable").to_csv(OUT_DIR / "fe_bh_table.csv", index=False)

## Interpretation guide

**Order of checking**

1. Only combinations with `reject_bh = True` are accepted as a "statistically stable signal"
2. If one of them has `small_cell = True` (low x TypeC, n=28), the sample is small and the coefficient may be unstable, so
   describe it with one notch less confidence even if it passes BH
3. For combinations that survive in a tier whose `pretrend_note` says "borderline" or "low power", always add the caveat that
   the "campaign effect" and the "pre-existing pre-trend" were not fully separated
4. **Take particular care if a significant result appears in the high tier**: in step 5 the high tier
   had only 49 non-recipient households, so the parallel pre-trend test itself had little power

## Next step
Take the type x tier combinations finalized here to step 8 (placebo test, 916 households that never received a campaign).
Only if the same pattern does not appear in the placebo test can we say "it may be a campaign effect".